<a href="https://colab.research.google.com/github/RobertFlan02/Dietetics-FYP/blob/main/Dietetics-FYP/Experiments/YOLO/Experiment%201/YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
#!/usr/bin/env python
# coding: utf-8

import os
import io
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

# ------------------------------
# Configuration: Update these paths as needed
# ------------------------------
# List of parquet files (train and validation)
TRAIN_PARQUET_PATHS = [
    "/content/drive/MyDrive/foodseg103/train-00000-of-00003-6bb37ec387d1825a.parquet",
    "/content/drive/MyDrive/foodseg103/train-00001-of-00003-4a1caa37147c0681.parquet",
    "/content/drive/MyDrive/foodseg103/train-00002-of-00003-c8b698399244cd95.parquet"
]

VAL_PARQUET_PATHS = [
    "/content/drive/MyDrive/foodseg103/validation-00000-of-00001-a5bfdaa5beb7006a.parquet"
]

# Output directories for YOLO training data
OUTPUT_DIR = "/content/drive/MyDrive/food_yolo"
TRAIN_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images/train")
TRAIN_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels/train")
VAL_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images/val")
VAL_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels/val")

# Padding in pixels to add around each bounding box (to preserve context)
PADDING = 10

# Background class value (assumed to be 0)
BACKGROUND_CLASS = 0

# ------------------------------
# Helper Functions
# ------------------------------

def load_parquet_files(parquet_paths):
    """Load and concatenate parquet files into a single DataFrame."""
    dfs = []
    for path in parquet_paths:
        try:
            df = pd.read_parquet(path)
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {path}: {e}")
    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        return full_df
    else:
        return pd.DataFrame()

def extract_image_and_mask(row):
    """
    Extracts an image and its corresponding mask from a DataFrame row.
    Expects row['image'] and row['label'] to be dictionaries with key "bytes".
    """
    # Extract image bytes and open as RGB image
    img_bytes = row['image'].get("bytes")
    if img_bytes is None:
        raise ValueError("No image bytes found.")
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    # Extract mask bytes and open (assumed grayscale or palette)
    mask_bytes = row['label'].get("bytes")
    if mask_bytes is None:
        raise ValueError("No mask bytes found.")
    mask = Image.open(io.BytesIO(mask_bytes))
    return img, mask

def compute_bounding_boxes(mask):
    """
    Compute bounding boxes for each connected component in the mask.
    Returns a list of tuples: (class_id, x, y, w, h).
    """
    mask_np = np.array(mask)
    boxes = []
    # Get unique class labels in the mask
    for cls in np.unique(mask_np):
        if cls == BACKGROUND_CLASS:
            continue
        # Create binary mask for the current class
        binary = np.uint8(mask_np == cls) * 255
        # Find contours using RETR_EXTERNAL to get external contours only
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < 10:
                # Ignore very small regions
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            boxes.append((int(cls), x, y, w, h))
    return boxes

def convert_bbox_to_yolo(box, img_width, img_height, padding=PADDING):
    cls, x, y, w, h = box
    new_cls = cls - 1  # adjust food class label
    # Add padding (as before)
    x_padded = max(0, x - padding)
    y_padded = max(0, y - padding)
    x2_padded = min(img_width, x + w + padding)
    y2_padded = min(img_height, y + h + padding)
    w_padded = x2_padded - x_padded
    h_padded = y2_padded - y_padded
    x_center = (x_padded + w_padded / 2) / img_width
    y_center = (y_padded + h_padded / 2) / img_height
    w_norm = w_padded / img_width
    h_norm = h_padded / img_height
    return new_cls, x_center, y_center, w_norm, h_norm

def save_yolo_annotation(image_id, boxes, dest_folder):
    """
    Save YOLO formatted annotation file for a given image.
    Each line: <class> <x_center> <y_center> <width> <height>
    """
    txt_path = os.path.join(dest_folder, f"{image_id}.txt")
    with open(txt_path, "w") as f:
        for box in boxes:
            cls, x_center, y_center, w_norm, h_norm = box
            f.write(f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

def process_parquet_to_yolo(parquet_paths, images_dest, labels_dest):
    """
    Process each row of the concatenated DataFrame from parquet files:
      - Extract image and mask.
      - Compute bounding boxes for each object.
      - Convert each box to YOLO format with normalization.
      - Save the image and a corresponding text file with annotations.
    """
    df = load_parquet_files(parquet_paths)
    if df.empty:
        print("No data loaded from the given parquet files.")
        return

    os.makedirs(images_dest, exist_ok=True)
    os.makedirs(labels_dest, exist_ok=True)

    print(f"Processing {len(df)} samples...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting data"):
        try:
            img, mask = extract_image_and_mask(row)
            # Use a unique identifier; if row has an 'id', use it; else use idx.
            image_id = str(row.get("id", idx))
            # Save image as JPEG
            img_save_path = os.path.join(images_dest, f"{image_id}.jpg")
            img.save(img_save_path, format="JPEG")
            # Get image dimensions for normalization
            img_width, img_height = img.size
            # Compute bounding boxes from the mask
            raw_boxes = compute_bounding_boxes(mask)
            if not raw_boxes:
                # If no objects found, optionally write an empty label file
                open(os.path.join(labels_dest, f"{image_id}.txt"), "w").close()
                continue
            # Convert each bounding box to YOLO format with padding
            yolo_boxes = [convert_bbox_to_yolo(box, img_width, img_height, padding=PADDING)
                          for box in raw_boxes]
            # Save YOLO annotations
            save_yolo_annotation(image_id, yolo_boxes, labels_dest)
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")

# ------------------------------
# Main Execution
# ------------------------------
if __name__ == "__main__":
    # Process training data
    print("Processing training data...")
    process_parquet_to_yolo(
        TRAIN_PARQUET_PATHS,
        images_dest=TRAIN_IMAGES_DIR,
        labels_dest=TRAIN_LABELS_DIR
    )

    # Process validation data
    print("Processing validation data...")
    process_parquet_to_yolo(
        VAL_PARQUET_PATHS,
        images_dest=VAL_IMAGES_DIR,
        labels_dest=VAL_LABELS_DIR
    )

    print("Conversion complete. Check your output directories for images and YOLO annotation files.")


Processing training data...
Processing 4983 samples...


Converting data: 100%|██████████| 4983/4983 [03:58<00:00, 20.88it/s]


Processing validation data...
Processing 2135 samples...


Converting data: 100%|██████████| 2135/2135 [05:58<00:00,  5.96it/s]

Conversion complete. Check your output directories for images and YOLO annotation files.


In [10]:
import pandas as pd
import yaml

csv_path = "/content/drive/MyDrive/foodseg103/class_mappings.csv"
df = pd.read_csv(csv_path)

# Only include rows where Class Id > 0 (i.e., ignore background)
df = df[df["Class Id"] > 0]

# Create names dictionary by re-indexing: new_id = original_id - 1
names = {}
for _, row in df.iterrows():
    new_id = int(row["Class Id"]) - 1  # shift food labels down by 1
    names[new_id] = row["Class Name"]

dataset_yaml = {
    "path": "/content/drive/MyDrive/food_yolo",
    "train": "images/train",
    "val": "images/val",
    "names": names
}

yaml_path = "/content/drive/MyDrive/food_yolo/food_yolo.yaml"
with open(yaml_path, 'w') as file:
    yaml.dump(dataset_yaml, file, default_flow_style=False)

print(f"YAML file saved to: {yaml_path}")


YAML file saved to: /content/drive/MyDrive/food_yolo/food_yolo.yaml


In [3]:
!pip install ultralytics --upgrade


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.8/949.8 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 79.6 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [11]:
!yolo task=detect mode=train model=yolov8m.pt data=/content/drive/MyDrive/food_yolo/food_yolo.yaml epochs=50 imgsz=640 project=/content/drive/MyDrive/food_yolo/results name=yolov8m_detect

Ultralytics 8.3.95 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: task=detect, mode=train, model=yolov8m.pt, data=/content/drive/MyDrive/food_yolo/food_yolo.yaml, epochs=50, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/food_yolo/results, name=yolov8m_detect3, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False, save_frames=False, save_txt=False, save_conf=F

In [14]:
!yolo task=detect mode=predict model=/content/drive/MyDrive/food_yolo/results/yolov8m_detect3/weights/best.pt source=/content/drive/MyDrive/food_yolo/images/val name=yolov8m_detect_infer

Ultralytics 8.3.95 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 92 layers, 25,899,397 parameters, 0 gradients, 79.0 GFLOPs

image 1/2135 /content/drive/MyDrive/food_yolo/images/val/0.jpg: 608x640 2 dried cranberriess, 1 chicken duck, 1 shrimp, 1 rice, 1 carrot, 1 French beans, 40.1ms
image 2/2135 /content/drive/MyDrive/food_yolo/images/val/1.jpg: 608x640 1 shrimp, 1 carrot, 1 green beans, 36.8ms
image 3/2135 /content/drive/MyDrive/food_yolo/images/val/10.jpg: 480x640 1 chicken duck, 1 sauce, 1 bread, 1 rice, 1 pie, 2 potatos, 52.3ms
image 4/2135 /content/drive/MyDrive/food_yolo/images/val/100.jpg: 640x480 1 pineapple, 1 chicken duck, 1 sauce, 1 tomato, 39.5ms
image 5/2135 /content/drive/MyDrive/food_yolo/images/val/1000.jpg: 640x640 2 cheese butters, 1 coffee, 1 sauce, 1 bread, 1 pie, 37.9ms
image 6/2135 /content/drive/MyDrive/food_yolo/images/val/1001.jpg: 384x640 1 sauce, 2 rices, 1 garlic, 1 carrot, 1 broccoli, 38.7ms
image 7/2135 /content/dr